In [1]:
%load_ext autoreload
%autoreload 2

import os
import torch

torch.set_float32_matmul_precision("medium")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TORCHDYNAMO_DISABLE"] = "1"  # avoid triton/inductor version mismatch

## Configuration
Set the wandb run ID and MTEB tasks to evaluate on. The task type (ae/declutr) is auto-detected from the wandb config.

In [10]:
# === Configure these ===
WANDB_ID = "g84cae2b"  # wandb run ID
DEVICE = "cuda"
BATCH_SIZE = 64

# MTEB tasks to run (a representative subset)
MTEB_TASKS = [
    # STS
    #"STSBenchmark",
    #"SICK-R",
    # Classification
    #"Banking77Classification.v2",
    # "YahooAnswersTopicsClassification"
     #"TweetTopicSingleClassification",
    "MassiveIntentClassification",
    #"ToxicConversationsClassification",
    # Clustering
    #"ArXivHierarchicalClusteringS2S",
    #"RedditClustering",
    # Retrieval
    #"SciFact",
    #"NFCorpus",
]

## Load encoder from checkpoint

In [11]:
from mteb_wrapper import MTEBEncoderWrapper, load_encoder_from_checkpoint

encoder = load_encoder_from_checkpoint(
    wandb_id=WANDB_ID,
    device=DEVICE,
)

print(f"Encoder loaded: backbone={encoder.cfg.model_name}")
if encoder.cfg.sem is not None:
    print(f"SEM config: L={encoder.sem.cfg.L}, V={encoder.sem.cfg.V}")
    print(f"Embedding dim (L*V): {encoder.sem.cfg.L * encoder.sem.cfg.V}")

Detected task type: distill_simcse


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: FacebookAI/roberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Encoder loaded: backbone=FacebookAI/roberta-large
SEM config: L=2048, V=16
Embedding dim (L*V): 32768


In [29]:
from model.encoder import EncoderModel, EncoderConfig

encoder = EncoderModel(EncoderConfig(
    model_name="FacebookAI/roberta-large",
    no_out_proj=True,
))

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: FacebookAI/roberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Run MTEB evaluation
Evaluates in both **soft** (standard temperature) and **hard** (near-zero temperature, one-hot) modes.

In [15]:
import mteb
from copy import deepcopy

# Get only the english part of the tasks (some tasks have multiple languages, but we only want to evaluate on english)
tasks = mteb.get_tasks(tasks=MTEB_TASKS, languages=["eng"])
results = {}
modes = ["soft", "hard"] if encoder.cfg.sem is not None else ["soft"]  # only evaluate "soft" if SEM is present
for mode in modes:
    print(f"\n{'='*60}")
    print(f"Evaluating mode: {mode}")
    print(f"{'='*60}\n")

    model = MTEBEncoderWrapper(
        encoder=encoder,
        mode=mode,
        batch_size=512,
        device=DEVICE
    )

    results[mode] = mteb.evaluate(model=model, tasks=deepcopy(tasks))
print("\nDone!")


Evaluating mode: soft



Evaluating tasks:   0%|          | 0/1 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/51 [00:00<?, ?it/s]

/home/leog/scratch/sentence_diffusion/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/leog/scratch/sentence_diffusion/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/leog/scratch/sentence_diffusion/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c


Done!


## Compare soft vs hard results

In [13]:
import pandas as pd


def extract_main_score(task_results):
    """Extract the main score from MTEB task results."""
    scores = {}
    for task_result in task_results:
        task_name = task_result.task_name
        for split in ["test", "dev", "validation"]:
            if split in task_result.scores:
                score_list = task_result.scores[split]
                if score_list:
                    scores[task_name] = score_list[0].get("main_score", None)
                    break
    return scores


soft_scores = extract_main_score(results["soft"])
if "hard" in results:
    hard_scores = extract_main_score(results["hard"])
    df_dict = {
        "Task":  list(soft_scores.keys()),
        "Soft":  [soft_scores[t]     for t in soft_scores],
        "Hard":  [hard_scores.get(t) for t in soft_scores],
    }
else:
    df_dict = {
        "Task":  list(soft_scores.keys()),
        "Soft":  [soft_scores[t]     for t in soft_scores],
    }

df = pd.DataFrame(
    df_dict
).set_index("Task")

if "Hard" in df.columns:
    df["Delta"] = df["Soft"] - df["Hard"]

print(df.to_string(float_format="{:.4f}".format))
print(f"\nAverage soft:  {df['Soft'].mean():.4f}")
if "Hard" in df.columns:
    print(f"Average hard:  {df['Hard'].mean():.4f}")
    print(f"Average delta: {df['Delta'].mean():.4f}")

                              Soft   Hard   Delta
Task                                             
MassiveIntentClassification 0.6036 0.6059 -0.0023

Average soft:  0.6036
Average hard:  0.6059
Average delta: -0.0023


In [9]:
soft_scores = extract_main_score(results["soft"])
if "hard" in results:
    hard_scores = extract_main_score(results["hard"])
    df_dict = {
        "Task":  list(soft_scores.keys()),
        "Soft":  [soft_scores[t]     for t in soft_scores],
        "Hard":  [hard_scores.get(t) for t in soft_scores],
    }
else:
    df_dict = {
        "Task":  list(soft_scores.keys()),
        "Soft":  [soft_scores[t]     for t in soft_scores],
    }

df = pd.DataFrame(
    df_dict
).set_index("Task")

if "Hard" in df.columns:
    df["Delta"] = df["Soft"] - df["Hard"]

print(df.to_string(float_format="{:.4f}".format))
print(f"\nAverage soft:  {df['Soft'].mean():.4f}")
if "Hard" in df.columns:
    print(f"Average hard:  {df['Hard'].mean():.4f}")
    print(f"Average delta: {df['Delta'].mean():.4f}")

               Soft   Hard   Delta
Task                              
STSBenchmark 0.8780 0.8805 -0.0024

Average soft:  0.8780
Average hard:  0.8805
Average delta: -0.0024


In [9]:
soft_scores = extract_main_score(results["soft"])
if "hard" in results:
    hard_scores = extract_main_score(results["hard"])
    df_dict = {
        "Task":  list(soft_scores.keys()),
        "Soft":  [soft_scores[t]     for t in soft_scores],
        "Hard":  [hard_scores.get(t) for t in soft_scores],
    }
else:
    df_dict = {
        "Task":  list(soft_scores.keys()),
        "Soft":  [soft_scores[t]     for t in soft_scores],
    }

df = pd.DataFrame(
    df_dict
).set_index("Task")

if "Hard" in df.columns:
    df["Delta"] = df["Soft"] - df["Hard"]

print(df.to_string(float_format="{:.4f}".format))
print(f"\nAverage soft:  {df['Soft'].mean():.4f}")
if "Hard" in df.columns:
    print(f"Average hard:  {df['Hard'].mean():.4f}")
    print(f"Average delta: {df['Delta'].mean():.4f}")

               Soft   Hard   Delta
Task                              
STSBenchmark 0.8780 0.8805 -0.0024

Average soft:  0.8780
Average hard:  0.8805
Average delta: -0.0024


In [17]:
soft_scores = extract_main_score(results["soft"])
if "hard" in results:
    hard_scores = extract_main_score(results["hard"])
    df_dict = {
        "Task":  list(soft_scores.keys()),
        "Soft":  [soft_scores[t]     for t in soft_scores],
        "Hard":  [hard_scores.get(t) for t in soft_scores],
    }
else:
    df_dict = {
        "Task":  list(soft_scores.keys()),
        "Soft":  [soft_scores[t]     for t in soft_scores],
    }

df = pd.DataFrame(
    df_dict
).set_index("Task")

if "Hard" in df.columns:
    df["Delta"] = df["Soft"] - df["Hard"]

print(df.to_string(float_format="{:.4f}".format))
print(f"\nAverage soft:  {df['Soft'].mean():.4f}")
if "Hard" in df.columns:
    print(f"Average hard:  {df['Hard'].mean():.4f}")
    print(f"Average delta: {df['Delta'].mean():.4f}")

                                 Soft
Task                                 
STSBenchmark                   0.4701
Banking77Classification.v2     0.6705
MassiveIntentClassification    0.6171
ArXivHierarchicalClusteringS2S 0.5231
SciFact                        0.0795

Average soft:  0.4721
